In [ ]:
import polars as pl
import json
import yaml

with open("../config.yaml", "r") as fp:
    config = yaml.safe_load(fp)

In [ ]:
DIR = f"{config['dirs']['output']}/partitioned"

In [ ]:
try:
    with open("../output/partition_stats.json", "r") as fp:
        stats = json.load(fp)
except FileNotFoundError:
    # runs 5m 15s
    stats = {}
    for i in range(256):
        data = pl.read_parquet(f"{DIR}/{i:02x}/pings.parquet")
        stats[i] = {
            "number_of_records": len(data),
            "number_of_devices": len(data["device_id"].unique()),
        }
        del data
    with open("../output/partition_stats.json", "w") as fp:
        json.dump(stats, fp, indent=1)

In [17]:
records = []
for k, v in stats.items():
    records.append([k, v["number_of_records"], v["number_of_devices"]])
stats_df = pl.from_records(
    records,
    schema=["partition", "number_of_records", "number_of_devices"],
    orient="row",
)

In [32]:
stats_df["number_of_records"].sum()

2860417726

In [30]:
print(
    stats_df.describe()
    .drop("partition")
    .with_columns(
        pl.col.number_of_records.round().cast(pl.Int64),
        pl.col.number_of_devices.round().cast(pl.Int64),
    )
    .to_pandas()
    .to_markdown(index=False)
)

| statistic   |   number_of_records |   number_of_devices |
|:------------|--------------------:|--------------------:|
| count       |                 256 |                 256 |
| null_count  |                   0 |                   0 |
| mean        |            11173507 |                6455 |
| std         |             2886655 |                 115 |
| min         |             6552223 |                6153 |
| 25%         |             8822529 |                6382 |
| 50%         |             9466836 |                6459 |
| 75%         |            14584310 |                6536 |
| max         |            16702601 |                6765 |
